# Phase 2 — CLV Modelling (BG/NBD + Gamma-Gamma)

Fits probabilistic CLV models and produces 90-day and 365-day scores for every repeat customer.

**Sections**
1. Setup & data preparation
2. BG/NBD model fitting & diagnostics
3. Gamma-Gamma assumption check & fitting
4. CLV scoring & distributions
5. CLV vs actual revenue
6. CLV segment analysis
7. Summary

## 1. Setup & Data Preparation

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.plotting import plot_frequency_recency_matrix, plot_probability_alive_matrix
from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, build_master_customer_table
)
from models.clv_model import (
    prepare_clv_data, fit_bgnbd, predict_purchases,
    check_gg_assumption, fit_gamma_gamma, score_clv,
    save_models, build_clv_scores
)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df_raw = load_raw_data()
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, _ = clean_data(df_raw)
master = build_master_customer_table(df_customers, return_features, cancel_features)
clv_df = prepare_clv_data(master)
print(clv_df[['frequency_repeat', 'recency_bgnbd', 'T_bgnbd', 'monetary']].describe().round(2))

## 2. BG/NBD Model Fitting & Diagnostics

In [ ]:
bgf = fit_bgnbd(clv_df)
predicted_purchases_90d, prob_alive = predict_purchases(bgf, clv_df)

---
### Plot 1 — Frequency × Recency Matrix
**Business question: For a given recency and frequency, how many purchases does the model expect in the next 90 days?**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_frequency_recency_matrix(bgf, T=90, ax=ax)
ax.set_title('BG/NBD — Expected Purchases in Next 90 Days')
plt.tight_layout()
plt.show()

---
### Plot 2 — Probability Alive Matrix
**Business question: What is the probability that a customer with a given recency and frequency is still active?**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_probability_alive_matrix(bgf, ax=ax)
ax.set_title('BG/NBD — Probability Customer is Still Alive')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(prob_alive, bins=60, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_title('Distribution of P(Alive) Across Repeat Customers')
ax.set_xlabel('Probability Alive')
ax.legend()
pct_alive = (prob_alive > 0.5).mean() * 100
ax.text(0.55, ax.get_ylim()[1] * 0.85, f'{pct_alive:.1f}% of customers\nhave P(alive) > 0.5', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Gamma-Gamma Assumption Check & Fitting

---
### Assumption Check — Frequency vs Monetary Independence
**Business question: Is the Gamma-Gamma model's independence assumption satisfied for this dataset?**

In [ ]:
corr = check_gg_assumption(clv_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
freq_clip = clv_df['frequency_repeat'].clip(upper=clv_df['frequency_repeat'].quantile(0.97))
mon_clip = clv_df['monetary'].clip(upper=clv_df['monetary'].quantile(0.97))
ax.scatter(freq_clip, mon_clip, alpha=0.2, s=8, color='steelblue')
ax.set_xlabel('Frequency (repeat purchases)')
ax.set_ylabel('Monetary (mean order £)')
ax.set_title(f'Frequency vs Monetary  (Pearson r = {corr:.3f})')
if abs(corr) > 0.3:
    ax.text(0.05, 0.92, '⚠ |r| > 0.3 — independence assumption may be violated',
            transform=ax.transAxes, color='red', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
ggf = fit_gamma_gamma(clv_df)